# Stage 4: 馬券最適化 強化学習（PPO + MCTS）訓練・推論ノートブック

**AREA-07 §10** で設計した馬券選択モデリング基盤 Phase 5b/5c の実装・訓練・推論テスト。

| § | 内容 |
|---|---|
| 1 | モックデータ生成 |
| 2 | EV計算エンジン（Phase 5a: Harville + Kelly） |
| 3 | BettingTransformer（Phase 5b: DL価値ネットワーク） |
| 4 | PPO Actor-Critic（Phase 5c: RLポリシーネットワーク） |
| 5 | PPO訓練ループ（モックエピソード） |
| 6 | ONNXエクスポート |
| 7 | ONNX推論テスト（onnxruntime / CPU） |
| 8 | BettingMCTS（Phase 5c: UCB1 + 価値ネット） |
| 9 | フルパイプライン動作確認 |


In [1]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import onnxruntime as ort
import math, os, time, warnings
from dataclasses import dataclass, field
from typing import List, Dict, Tuple
from collections import defaultdict

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# ─── Config ───────────────────────────────────────────────────────────────
N_HORSES      = 18
INPUT_DIM     = 8      # 馬ごとの特徴数
D_MODEL       = 64     # Transformer 隠れ次元
N_HEADS       = 4
N_LAYERS      = 2
TOP_K         = 5      # 候補生成に使う上位 K 頭（固定アクション空間）
N_WIN         = TOP_K
N_PLACE       = TOP_K
N_QUI         = TOP_K * (TOP_K - 1) // 2   # C(5,2) = 10
N_CAND        = N_WIN + N_PLACE + N_QUI     # 20
RACE_FEAT_DIM = 4
STATE_DIM     = D_MODEL + RACE_FEAT_DIM     # 68
CAND_FEAT_DIM = 3
FULL_DIM      = STATE_DIM + N_CAND * CAND_FEAT_DIM  # 128
N_EPISODES    = 400
BATCH_SIZE    = 32
PPO_EPOCHS    = 4
PPO_CLIP      = 0.2
LR            = 3e-4
KELLY_FRAC    = 0.25
ONNX_DIR      = "/tmp/keiba_rl_models"
os.makedirs(ONNX_DIR, exist_ok=True)

device = torch.device("cpu")   # VPS 互換: CPU のみ
print(f"torch={torch.__version__}, ort={ort.__version__}, device={device}")
print(f"N_CAND={N_CAND}, FULL_DIM={FULL_DIM}")


torch=2.11.0+cu130, ort=1.24.4, device=cpu
N_CAND=20, FULL_DIM=128


## §1. モックデータ生成

In [2]:
def generate_mock_race(n: int = N_HORSES, seed: int = None) -> Dict:
    rng = np.random.RandomState(seed)
    win_prob = rng.dirichlet(np.ones(n) * 0.5)
    place_prob = np.clip(win_prob * rng.uniform(1.4, 2.4, n), 0, 1.0)
    place_prob = np.maximum(place_prob, win_prob)
    if place_prob.sum() > 2.0:
        place_prob = place_prob / place_prob.sum() * 2.0
    show_prob = np.clip(place_prob * rng.uniform(1.1, 1.5, n), 0, 1.0)
    show_prob = np.maximum(show_prob, place_prob)
    if show_prob.sum() > 3.0:
        show_prob = show_prob / show_prob.sum() * 3.0
    # 市場オッズは「市場のインプライド確率」から算出（モデルと乖離あり）
    # → 一部の馬で model_win_prob > market_implied → EV > 1.0 が発生
    market_win = np.clip(win_prob * rng.lognormal(0, 0.5, n), 1e-9, 1.0)
    market_win /= market_win.sum()
    market_place = np.clip(place_prob * rng.lognormal(0, 0.3, n), 1e-9, 1.0)
    market_place = np.minimum(market_place, 2.0)
    # 日本競馬の控除率 ~25%: payout_rate = 0.75
    odds_win   = np.clip(0.75 / np.maximum(market_win,   1e-6), 1.1, 150.0)
    odds_place = np.clip(0.75 / np.maximum(market_place, 1e-6), 1.05, 30.0)
    return dict(
        race_id=f"mock_{(seed or 0):04d}", n_horses=n,
        horse_ids=[f"h{i+1:02d}" for i in range(n)],
        win_prob=win_prob, place_prob=place_prob, show_prob=show_prob,
        odds_win=odds_win, odds_place=odds_place,
        pace=rng.randn(n), form=rng.uniform(0, 1, n),
    )

def build_horse_features(race: Dict) -> np.ndarray:
    return np.stack([
        race['win_prob'], race['place_prob'], race['show_prob'],
        np.log(race['odds_win']), np.log(race['odds_place']),
        race['win_prob'] * race['odds_win'],   # implied EV
        race['pace'], race['form'],
    ], axis=1).astype(np.float32)  # (N, 8)

race0 = generate_mock_race(seed=42)
f0 = build_horse_features(race0)
print(f"race0: n_horses={race0['n_horses']}, win_prob sum={race0['win_prob'].sum():.4f}")
print(f"horse_features shape: {f0.shape}")
print(f"win_prob top-5: {np.sort(race0['win_prob'])[::-1][:5].round(3)}")
print(f"odds_win  top-5 (cheapest): {np.sort(race0['odds_win'])[:5].round(1)}")


race0: n_horses=18, win_prob sum=1.0000
horse_features shape: (18, 8)
win_prob top-5: [0.47  0.152 0.092 0.055 0.054]
odds_win  top-5 (cheapest): [ 1.6  6.1  7.4 10.9 13.9]


## §2. EV計算エンジン（Phase 5a: Harville + Kelly）

In [3]:
@dataclass
class BetCandidate:
    bet_type:     str
    horse_idx:    List[int]
    ev_score:     float = 0.0
    hit_prob:     float = 0.0
    implied_odds: float = 0.0
    kelly_frac:   float = 0.0

def kelly_fraction(ev: float, hit_prob: float, odds: float) -> float:
    if ev <= 1.0 or odds <= 1.0:
        return 0.0
    return float(np.clip((ev - 1.0) / (odds - 1.0) * KELLY_FRAC, 0.0, 0.25))

def build_bet_candidates(race: Dict) -> List[BetCandidate]:
    wp, pp = race['win_prob'], race['place_prob']
    ow, op = race['odds_win'], race['odds_place']
    top = np.argsort(wp)[::-1][:TOP_K].tolist()
    cands: List[BetCandidate] = []
    for i in top:
        ev = float(wp[i] * ow[i])
        c = BetCandidate('WIN', [i], ev, float(wp[i]), float(ow[i]))
        c.kelly_frac = kelly_fraction(ev, c.hit_prob, c.implied_odds)
        cands.append(c)
    for i in top:
        ev = float(pp[i] * op[i])
        c = BetCandidate('PLACE', [i], ev, float(pp[i]), float(op[i]))
        c.kelly_frac = kelly_fraction(ev, c.hit_prob, c.implied_odds)
        cands.append(c)
    for ii in range(len(top)):
        for jj in range(ii + 1, len(top)):
            i, j = top[ii], top[jj]
            q_p = float(wp[i] * wp[j] / max(1 - wp[i], 1e-9) +
                        wp[j] * wp[i] / max(1 - wp[j], 1e-9))
            q_p = min(q_p, 0.99)
            q_o = float(np.clip(0.8 / max(q_p, 1e-6), 1.5, 200.0))
            ev  = float(q_p * q_o)
            c = BetCandidate('QUINELLA', [i, j], ev, q_p, q_o)
            c.kelly_frac = kelly_fraction(ev, q_p, q_o)
            cands.append(c)
    assert len(cands) == N_CAND
    return cands

cands0 = build_bet_candidates(race0)
print(f"候補数: {len(cands0)}  ({N_WIN} WIN + {N_PLACE} PLACE + {N_QUI} QUINELLA)")
print("\nEV 上位 5 件:")
for c in sorted(cands0, key=lambda x: x.ev_score, reverse=True)[:5]:
    names = "+".join(race0['horse_ids'][k] for k in c.horse_idx)
    print(f"  {c.bet_type:10s} {names:12s}  EV={c.ev_score:.3f}  "
          f"hit={c.hit_prob:.3f}  odds={c.implied_odds:.1f}  kelly={c.kelly_frac:.4f}")


候補数: 20  (5 WIN + 5 PLACE + 10 QUINELLA)

EV 上位 5 件:
  WIN        h07           EV=1.130  hit=0.152  odds=7.4  kelly=0.0050
  PLACE      h17           EV=0.985  hit=0.938  odds=1.1  kelly=0.0000
  QUINELLA   h17+h07       EV=0.800  hit=0.219  odds=3.7  kelly=0.0000
  QUINELLA   h17+h02       EV=0.800  hit=0.128  odds=6.2  kelly=0.0000
  QUINELLA   h17+h11       EV=0.800  hit=0.075  odds=10.6  kelly=0.0000


## §3. BettingTransformer（Phase 5b: DL価値ネットワーク）

In [4]:
class BettingTransformer(nn.Module):
    """AREA-07 §10-4 / d_model=64, nhead=4, 2 layers"""
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(INPUT_DIM, D_MODEL)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL, nhead=N_HEADS,
            dim_feedforward=256, dropout=0.1,
            batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=N_LAYERS)
        self.head    = nn.Linear(D_MODEL, 1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, N, INPUT_DIM) -> (B, N)
        h = F.relu(self.proj(x))
        h = self.encoder(h)
        return self.head(h).squeeze(-1)

    def encode_mean(self, x: torch.Tensor) -> torch.Tensor:
        # mean-pool -> (B, D_MODEL)
        h = F.relu(self.proj(x))
        return self.encoder(h).mean(dim=1)

transformer = BettingTransformer()
n_bt = sum(p.numel() for p in transformer.parameters())
print(f"BettingTransformer params: {n_bt:,}  (~{n_bt*4//1024} KB)")

xtest = torch.tensor(f0).unsqueeze(0)  # (1, 18, 8)
with torch.no_grad():
    ev_out = transformer(xtest)          # (1, 18)
    enc    = transformer.encode_mean(xtest)  # (1, 64)
print(f"ev_out shape: {ev_out.shape},  encode_mean shape: {enc.shape}")
print(f"ev top-5: {ev_out[0].topk(5).values.detach().numpy().round(3)}")


BettingTransformer params: 100,609  (~393 KB)
ev_out shape: torch.Size([1, 18]),  encode_mean shape: torch.Size([1, 64])
ev top-5: [ 0.004 -0.179 -0.322 -0.327 -0.346]


## §4. PPO Actor-Critic（Phase 5c: RLポリシーネットワーク）

In [5]:
def _mlp(in_d: int, arch: List[int], out_d: int) -> nn.Sequential:
    layers, cur = [], in_d
    for h in arch:
        layers += [nn.Linear(cur, h), nn.LayerNorm(h), nn.ReLU()]
        cur = h
    layers.append(nn.Linear(cur, out_d))
    return nn.Sequential(*layers)

class PPOActor(nn.Module):
    """AREA-07 §10-4 Actor arch [256,128,64]  input=FULL_DIM  output=N_CAND logits"""
    def __init__(self):
        super().__init__()
        self.net = _mlp(FULL_DIM, [256, 128, 64], N_CAND)
    def forward(self, x): return self.net(x)

class PPOCritic(nn.Module):
    """AREA-07 §10-4 Critic arch [256,128,1]  input=FULL_DIM  output scalar V(s)"""
    def __init__(self):
        super().__init__()
        self.net = _mlp(FULL_DIM, [256, 128, 64], 1)
    def forward(self, x): return self.net(x).squeeze(-1)

def build_full_state(race: Dict, xformer: BettingTransformer,
                     cands: List[BetCandidate]) -> np.ndarray:
    feats = torch.tensor(build_horse_features(race)).unsqueeze(0)
    with torch.no_grad():
        enc = xformer.encode_mean(feats)[0].numpy()   # (64,)
    race_f = np.array([
        race['n_horses'] / 18.0,
        float(np.log(race['odds_win'].mean())),
        float(race['win_prob'].std()),
        float(race['win_prob'].max()),
    ], dtype=np.float32)
    cand_f = np.array(
        [[c.ev_score, c.hit_prob, np.log(max(c.implied_odds, 1.0))] for c in cands],
        dtype=np.float32
    ).flatten()
    return np.concatenate([enc, race_f, cand_f]).astype(np.float32)  # (128,)

actor  = PPOActor()
critic = PPOCritic()
print(f"PPOActor  params: {sum(p.numel() for p in actor.parameters()):,}")
print(f"PPOCritic params: {sum(p.numel() for p in critic.parameters()):,}")

sv0 = build_full_state(race0, transformer, cands0)
print(f"full_state shape: {sv0.shape}")
with torch.no_grad():
    lg = actor(torch.tensor(sv0).unsqueeze(0))
    vv = critic(torch.tensor(sv0).unsqueeze(0))
print(f"actor logits: {lg.shape},  critic value: {vv.item():.4f}")


PPOActor  params: 76,372
PPOCritic params: 75,137
full_state shape: (128,)
actor logits: torch.Size([1, 20]),  critic value: 0.4729


## §5. PPO訓練ループ（モックエピソード）

In [6]:
def simulate_finish(race: Dict) -> Tuple[int, int, int]:
    wp = race['win_prob'].astype(np.float64); wp /= wp.sum()
    w = int(np.random.choice(len(wp), p=wp))
    wp2 = wp.copy(); wp2[w] = 0; wp2 /= wp2.sum()
    s = int(np.random.choice(len(wp2), p=wp2))
    wp3 = wp2.copy(); wp3[s] = 0; wp3 /= wp3.sum()
    t = int(np.random.choice(len(wp3), p=wp3))
    return w, s, t

def compute_roi(cands: List[BetCandidate], acts: np.ndarray,
                w: int, s: int, t: int) -> float:
    stake = payout = 0.0
    top2 = {w, s}
    for c, a in zip(cands, acts):
        if a < 0.5: continue
        stake += 1.0
        if c.bet_type == 'WIN'      and c.horse_idx[0] == w:       payout += c.implied_odds
        elif c.bet_type == 'PLACE'  and c.horse_idx[0] in top2:    payout += c.implied_odds
        elif c.bet_type == 'QUINELLA' and set(c.horse_idx[:2])==top2: payout += c.implied_odds
    return (payout - stake) / stake if stake else -0.5

actor_opt = optim.Adam(actor.parameters(), lr=LR)
critic_opt = optim.Adam(critic.parameters(), lr=LR)

buf_s, buf_a, buf_lp, buf_r = [], [], [], []
log_roi, log_aloss, log_closs = [], [], []

print("PPO 訓練開始 ...")
for ep in range(N_EPISODES):
    race = generate_mock_race(seed=ep)
    cands = build_bet_candidates(race)
    sv    = build_full_state(race, transformer, cands)
    st    = torch.tensor(sv).unsqueeze(0)

    with torch.no_grad():
        logits = actor(st)[0].numpy()
    probs  = 1.0 / (1.0 + np.exp(-logits))
    acts   = (np.random.rand(N_CAND) < probs).astype(np.float32)
    w, s, t = simulate_finish(race)
    roi    = compute_roi(cands, acts, w, s, t)

    eps = 1e-8
    lp  = float(np.where(acts > 0.5, np.log(probs + eps),
                         np.log(1 - probs + eps)).sum())
    buf_s.append(sv); buf_a.append(acts); buf_lp.append(lp); buf_r.append(roi)

    if len(buf_s) >= BATCH_SIZE:
        st_b  = torch.tensor(np.array(buf_s), dtype=torch.float32)
        ac_b  = torch.tensor(np.array(buf_a), dtype=torch.float32)
        olp_b = torch.tensor(buf_lp, dtype=torch.float32)
        rew_b = torch.tensor(buf_r,  dtype=torch.float32)

        with torch.no_grad():
            adv = rew_b - critic(st_b)

        for _ in range(PPO_EPOCHS):
            nlg  = actor(st_b)
            np_  = torch.sigmoid(nlg)
            nlp  = (ac_b * torch.log(np_ + 1e-8) +
                    (1 - ac_b) * torch.log(1 - np_ + 1e-8)).sum(1)
            rat  = torch.exp(nlp - olp_b)
            aloss = -torch.min(rat * adv.detach(),
                               torch.clamp(rat, 1-PPO_CLIP, 1+PPO_CLIP) * adv.detach()).mean()
            actor_opt.zero_grad(); aloss.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), 0.5)
            actor_opt.step()

            closs = F.mse_loss(critic(st_b), rew_b)
            critic_opt.zero_grad(); closs.backward(); critic_opt.step()

        log_roi.append(float(np.mean(buf_r)))
        log_aloss.append(float(aloss.item()))
        log_closs.append(float(closs.item()))
        buf_s.clear(); buf_a.clear(); buf_lp.clear(); buf_r.clear()

    if (ep + 1) % 100 == 0:
        roi_r = np.mean(log_roi[-3:]) if log_roi else 0
        print(f"  ep {ep+1:4d}/{N_EPISODES}  ROI={roi_r:.4f}  "
              f"aloss={log_aloss[-1]:.4f}  closs={log_closs[-1]:.4f}")

print("\n訓練完了")
print(f"  最終 mean ROI (last 5 batches): {np.mean(log_roi[-5:]):.4f}")


PPO 訓練開始 ...
  ep  100/400  ROI=-0.0907  aloss=-0.3122  closs=2.7045


  ep  200/400  ROI=-0.3167  aloss=-0.1692  closs=1.1552
  ep  300/400  ROI=-0.1546  aloss=-0.0355  closs=0.6493


  ep  400/400  ROI=-0.1650  aloss=-0.0276  closs=0.7219

訓練完了
  最終 mean ROI (last 5 batches): -0.1485


## §6. ONNXエクスポート

In [7]:
transformer.eval(); actor.eval()

# BettingTransformer
bt_path = os.path.join(ONNX_DIR, "betting_transformer.onnx")
torch.onnx.export(
    transformer,
    torch.randn(1, N_HORSES, INPUT_DIM),
    bt_path,
    dynamo=False,
    input_names=["horse_features"],
    output_names=["ev_scores"],
    dynamic_axes={"horse_features": {0: "batch", 1: "horses"},
                  "ev_scores":       {0: "batch", 1: "horses"}},
    opset_version=17,
)
print(f"BettingTransformer ONNX: {os.path.getsize(bt_path)/1024:.1f} KB  -> {bt_path}")

# PPO Actor
act_path = os.path.join(ONNX_DIR, "ppo_actor.onnx")
torch.onnx.export(
    actor,
    torch.randn(1, FULL_DIM),
    act_path,
    dynamo=False,
    input_names=["full_state"],
    output_names=["action_logits"],
    opset_version=17,
)
print(f"PPOActor ONNX:           {os.path.getsize(act_path)/1024:.1f} KB  -> {act_path}")

# PyTorch weights (再訓練用)
torch.save(transformer.state_dict(), os.path.join(ONNX_DIR, "betting_transformer.pt"))
torch.save(actor.state_dict(),       os.path.join(ONNX_DIR, "ppo_actor.pt"))
print(f"PyTorch weights saved to {ONNX_DIR}/")


BettingTransformer ONNX: 248.4 KB  -> /tmp/keiba_rl_models/betting_transformer.onnx
PPOActor ONNX:           300.1 KB  -> /tmp/keiba_rl_models/ppo_actor.onnx
PyTorch weights saved to /tmp/keiba_rl_models/


## §7. ONNX推論テスト（onnxruntime / CPU）

In [8]:
sess_bt  = ort.InferenceSession(bt_path,  providers=["CPUExecutionProvider"])
sess_act = ort.InferenceSession(act_path, providers=["CPUExecutionProvider"])

print("=== ONNX I/O シグネチャ ===")
for name, sess in [("BettingTransformer", sess_bt), ("PPOActor", sess_act)]:
    print(f"[{name}]")
    for i in sess.get_inputs():  print(f"  in : {i.name}  {i.shape}  {i.type}")
    for o in sess.get_outputs(): print(f"  out: {o.name}  {o.shape}  {o.type}")

# 推論テスト
race_t = generate_mock_race(seed=999)
ft = build_horse_features(race_t)[np.newaxis].astype(np.float32)

t0 = time.perf_counter()
ev_ort = sess_bt.run(None, {"horse_features": ft})[0]
t1 = time.perf_counter()
print(f"\nBettingTransformer ONNX: {(t1-t0)*1000:.2f} ms,  shape={ev_ort.shape}")
print(f"  ev_scores top-5: {np.sort(ev_ort[0])[::-1][:5].round(4)}")

cands_t = build_bet_candidates(race_t)
sv_t    = build_full_state(race_t, transformer, cands_t)[np.newaxis].astype(np.float32)
t0 = time.perf_counter()
logits_ort = sess_act.run(None, {"full_state": sv_t})[0]
t1 = time.perf_counter()
print(f"PPOActor ONNX:           {(t1-t0)*1000:.2f} ms,  shape={logits_ort.shape}")
probs_ort = (1.0 / (1.0 + np.exp(-logits_ort[0])))
print(f"  選択確率 > 0.5: {(probs_ort > 0.5).sum()} 件")

# PyTorch vs ONNX 差分
with torch.no_grad():
    ev_pt = transformer(torch.tensor(ft)).numpy()
diff = np.abs(ev_ort - ev_pt).max()
print(f"\nPyTorch vs ONNX 最大差分: {diff:.2e}  (許容 < 1e-4)")
assert diff < 1e-3, f"精度チェック失敗: diff={diff}"
print("精度チェック: PASS")


=== ONNX I/O シグネチャ ===
[BettingTransformer]
  in : horse_features  ['batch', 'horses', 8]  tensor(float)
  out: ev_scores  ['batch', 'horses']  tensor(float)
[PPOActor]
  in : full_state  [1, 128]  tensor(float)
  out: action_logits  [1, 20]  tensor(float)

BettingTransformer ONNX: 0.86 ms,  shape=(1, 18)
  ev_scores top-5: [-0.267  -0.4437 -0.4624 -0.4701 -0.4837]
PPOActor ONNX:           0.11 ms,  shape=(1, 20)
  選択確率 > 0.5: 10 件

PyTorch vs ONNX 最大差分: 5.96e-07  (許容 < 1e-4)
精度チェック: PASS


## §8. BettingMCTS（Phase 5c: UCB1 + ONNX価値ネット）

In [9]:
@dataclass
class _Node:
    sel:  List[BetCandidate] = field(default_factory=list)
    rem:  List[BetCandidate] = field(default_factory=list)
    vis:  int   = 0
    vsum: float = 0.0
    par:  object = None
    ch:   list   = field(default_factory=list)

class BettingMCTS:
    """AREA-07 §10-4 BettingMCTS  (n_simulations=50 推奨 for VPS 2GB)"""
    MAX_DEPTH = 5

    def __init__(self, cands: List[BetCandidate], sess_bt: ort.InferenceSession,
                 race: Dict, n_sim: int = 50, c_puct: float = 1.4):
        self.cands = cands; self.sess = sess_bt; self.race = race
        self.n_sim = n_sim; self.c_puct = c_puct
        ft = build_horse_features(race)[np.newaxis].astype(np.float32)
        self._ev = self.sess.run(None, {"horse_features": ft})[0][0]  # (N,)

    def _val(self, sel: List[BetCandidate]) -> float:
        if not sel: return 0.0
        # Transformer スコアは sigmoid で [0,1] に正規化してブースト係数として使用
        t_sig = 1.0 / (1.0 + np.exp(-self._ev))
        evs = []
        for c in sel:
            ev = c.ev_score
            if c.bet_type in ('WIN', 'PLACE') and c.horse_idx:
                boost = 0.8 + float(t_sig[c.horse_idx[0]]) * 0.4
                ev = ev * boost
            evs.append(ev)
        return float(np.mean(evs)) - 1.0

    def _ucb(self, n: _Node) -> float:
        if n.vis == 0: return float('inf')
        return n.vsum/n.vis + self.c_puct * math.sqrt(math.log(n.par.vis)/n.vis)

    def _sel(self, n: _Node) -> _Node:
        while n.ch: n = max(n.ch, key=self._ucb)
        if len(n.sel) < self.MAX_DEPTH and n.rem:
            for c in n.rem:
                n.ch.append(_Node(n.sel+[c],[x for x in n.rem if x is not c], par=n))
            n = n.ch[0]
        return n

    def _bp(self, n: _Node, v: float):
        while n: n.vis += 1; n.vsum += v; n = n.par

    def search(self) -> List[BetCandidate]:
        root = _Node(rem=list(self.cands))
        for _ in range(self.n_sim):
            leaf = self._sel(root); self._bp(leaf, self._val(leaf.sel))
        n = root
        while n.ch: n = max(n.ch, key=lambda x: x.vis)
        return n.sel

# テスト
race_m = generate_mock_race(seed=123)
cands_m = build_bet_candidates(race_m)
t0 = time.perf_counter()
mcts = BettingMCTS(cands_m, sess_bt, race_m, n_sim=50)
sel  = mcts.search()
mcts_elapsed_ms = (time.perf_counter() - t0) * 1000
print(f"MCTS (n_sim=50): {mcts_elapsed_ms:.1f} ms,  選択={len(sel)} 件")
print("\nMCTS 選択結果:")
for c in sel:
    names = "+".join(race_m['horse_ids'][k] for k in c.horse_idx)
    print(f"  {c.bet_type:10s} {names:12s}  EV={c.ev_score:.3f}  "
          f"hit={c.hit_prob:.3f}  odds={c.implied_odds:.1f}  kelly={c.kelly_frac:.4f}")


MCTS (n_sim=50): 2.0 ms,  選択=2 件

MCTS 選択結果:
  WIN        h18           EV=1.097  hit=0.360  odds=3.1  kelly=0.0119
  WIN        h10           EV=0.679  hit=0.090  odds=7.6  kelly=0.0000


## §9. フルパイプライン動作確認

In [10]:
def apply_balance_filter(cands: List[BetCandidate], acts: np.ndarray,
                          max_per_type: int = 2, k: int = 5) -> np.ndarray:
    out = acts.copy()
    for i, c in enumerate(cands):
        if out[i] < 0.5: continue
        if c.ev_score <= 1.05 or not (0.03 < c.hit_prob < 0.75) or c.kelly_frac <= 0.01:
            out[i] = 0
    # 馬券種上限 & 全体 k 件上限
    cnt: Dict[str, int] = defaultdict(int)
    order = sorted(range(len(cands)), key=lambda i: cands[i].ev_score, reverse=True)
    final = np.zeros(len(cands))
    picked = 0
    for i in order:
        if out[i] < 0.5: continue
        if cnt[cands[i].bet_type] < max_per_type and picked < k:
            final[i] = 1; cnt[cands[i].bet_type] += 1; picked += 1
    return final

def run_stage4(race: Dict, n_sim: int = 50) -> Dict:
    """Stage 4 推論 (AREA-07 §10-1-3): Step A → B → C → D"""
    t0 = time.perf_counter()
    ft = build_horse_features(race)[np.newaxis].astype(np.float32)

    # Step A: EV 計算
    cands = build_bet_candidates(race)

    # Step B: BettingTransformer DL スコアリング
    # Transformer は馬の相対的ランキングを出力（未訓練時は絶対EV値は信頼しない）
    # sigmoid で [0,1] に正規化し候補の EV に乗算ブースト（訓練後は重みを高めに調整）
    ev_ort = sess_bt.run(None, {"horse_features": ft})[0][0]  # (N_HORSES,)
    t_score = 1.0 / (1.0 + np.exp(-ev_ort))  # sigmoid → [0,1]
    for c in cands:
        if c.bet_type in ('WIN', 'PLACE') and c.horse_idx:
            boost = 0.8 + t_score[c.horse_idx[0]] * 0.4  # [0.8, 1.2] の乗算倍率
            c.ev_score = c.ev_score * boost

    # Step C: PPO でフィルタリング → MCTS で最終選択
    sv = build_full_state(race, transformer, cands)[np.newaxis].astype(np.float32)
    lg = sess_act.run(None, {"full_state": sv})[0][0]
    pr = 1.0 / (1.0 + np.exp(-lg))
    mcts_in = [c for c, p in zip(cands, pr) if p > 0.3] or cands
    selected = BettingMCTS(mcts_in, sess_bt, race, n_sim=n_sim).search()
    stage = "RL"

    # Step D: Kelly + バランスフィルタ
    acts = np.array([1.0 if c in selected else 0.0 for c in cands])
    filt = apply_balance_filter(cands, acts)
    recs = [c for c, f in zip(cands, filt) if f > 0.5]

    return {"race_id": race['race_id'], "stage": stage,
            "elapsed_ms": (time.perf_counter()-t0)*1000, "recs": recs}

print("=" * 62)
print(" Phase 5c フルパイプライン推論テスト  (5 レース)")
print("=" * 62)
for seed in range(5):
    r = generate_mock_race(seed=seed+200)
    res = run_stage4(r, n_sim=50)
    print(f"\n[{res['race_id']}]  stage={res['stage']}  "
          f"elapsed={res['elapsed_ms']:.1f} ms  推薦={len(res['recs'])} 件")
    for c in res['recs']:
        names = "+".join(r['horse_ids'][k] for k in c.horse_idx)
        print(f"  ✓ {c.bet_type:10s} {names:14s}  EV={c.ev_score:.3f}  "
              f"hit={c.hit_prob:.3f}  odds={c.implied_odds:.1f}  kelly={c.kelly_frac:.4f}")

# SLA ベンチマーク (20 レース)
print("\n[SLA ベンチマーク — 20 レース n_sim=50]")
elaps = []
for seed in range(20):
    r = generate_mock_race(seed=seed+500)
    t0 = time.perf_counter()
    run_stage4(r, n_sim=50)
    elaps.append((time.perf_counter()-t0)*1000)
print(f"  平均: {np.mean(elaps):.1f} ms  最大: {np.max(elaps):.1f} ms  "
      f"SLA(120,000ms): {'✓ PASS' if np.max(elaps) < 120000 else '✗ FAIL'}")

print("\n[訓練曲線サマリ]")
n_batch = len(log_roi)
print(f"  更新バッチ数: {n_batch}")
if n_batch >= 4:
    print(f"  ROI  first 4 batches: {np.mean(log_roi[:4]):.4f}")
    print(f"  ROI  last  4 batches: {np.mean(log_roi[-4:]):.4f}")


 Phase 5c フルパイプライン推論テスト  (5 レース)

[mock_0200]  stage=RL  elapsed=4.4 ms  推薦=0 件

[mock_0201]  stage=RL  elapsed=5.5 ms  推薦=1 件
  ✓ WIN        h18             EV=2.049  hit=0.136  odds=16.0  kelly=0.0196

[mock_0202]  stage=RL  elapsed=2.4 ms  推薦=0 件

[mock_0203]  stage=RL  elapsed=2.3 ms  推薦=0 件

[mock_0204]  stage=RL  elapsed=3.0 ms  推薦=0 件

[SLA ベンチマーク — 20 レース n_sim=50]
  平均: 2.9 ms  最大: 4.6 ms  SLA(120,000ms): ✓ PASS

[訓練曲線サマリ]
  更新バッチ数: 12
  ROI  first 4 batches: -0.0831
  ROI  last  4 batches: -0.1701


## サマリ

In [11]:
print("=" * 62)
print(" 実行確認サマリ")
print("=" * 62)
print(f"  BettingTransformer : {sum(p.numel() for p in transformer.parameters()):,} params")
print(f"  PPOActor           : {sum(p.numel() for p in actor.parameters()):,} params")
print(f"  PPOCritic          : {sum(p.numel() for p in critic.parameters()):,} params")
print()
print(f"  ONNX BettingTransformer : {os.path.getsize(bt_path)/1024:.1f} KB")
print(f"  ONNX PPOActor           : {os.path.getsize(act_path)/1024:.1f} KB")
print()
print(f"  PyTorch vs ONNX 差分    : {diff:.2e}")
print(f"  MCTS (n_sim=50) SLA     : {mcts_elapsed_ms:.1f} ms / レース  (上限 120,000 ms)")
print()
print("  デプロイ方針 (AREA-07 §10-11):")
print("    ローカル GPU で訓練 → ONNX エクスポート → VPS は onnxruntime のみ")
print("    月次再学習サイクル: 訓練(~15h GPU) → ONNX(<1MB) → rsync → restart")
print()
print("✓ 全コンポーネント動作確認完了")


 実行確認サマリ
  BettingTransformer : 100,609 params
  PPOActor           : 76,372 params
  PPOCritic          : 75,137 params

  ONNX BettingTransformer : 248.4 KB
  ONNX PPOActor           : 300.1 KB

  PyTorch vs ONNX 差分    : 5.96e-07
  MCTS (n_sim=50) SLA     : 2.0 ms / レース  (上限 120,000 ms)

  デプロイ方針 (AREA-07 §10-11):
    ローカル GPU で訓練 → ONNX エクスポート → VPS は onnxruntime のみ
    月次再学習サイクル: 訓練(~15h GPU) → ONNX(<1MB) → rsync → restart

✓ 全コンポーネント動作確認完了
